# Exercise 5 — MCPAgent

**MCPAgent** wires the MCP stack together: it holds a client, uses `select_mcp_tool` to route each query to the right tool, calls the tool via the client, and records the interaction in history.  Same class shape as all prior section agents.

In [ ]:
import json
from dataclasses import dataclass, field
@dataclass
class MCPToolDef:
    name: str
    description: str
    input_schema: dict = field(default_factory=dict)

def tool_schema_text(tools):
    lines = []
    for t in tools:
        params = ", ".join(t.input_schema.keys())
        lines.append("- " + t.name + "(" + params + "): " + t.description)
    return "\n".join(lines)
class MCPServer:
    def __init__(self, name="mcp_server"):
        self.name = name
        self._tools = {}
    def tool(self, name, description, schema=None):
        def _decorator(fn):
            self._tools[name] = {"def": MCPToolDef(name, description, schema or {}), "fn": fn}
            return fn
        return _decorator
    def list_tools(self):
        return [e["def"] for e in self._tools.values()]
    def call_tool(self, name, args):
        e = self._tools.get(name)
        if e is None:
            return "Error: unknown tool " + repr(name)
        try:
            return str(e["fn"](**args))
        except Exception as exc:
            return "Error: " + str(exc)
def call_llm(messages, llm_fn=None):
    if llm_fn is not None:
        return str(llm_fn(messages))
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]

def safe_parse_json(text):
    start = str(text).find("{")
    end   = str(text).rfind("}") + 1
    if start == -1 or end == 0:
        return None
    try:
        return json.loads(text[start:end])
    except (json.JSONDecodeError, ValueError):
        return None
class MCPClient:
    def __init__(self, server=None, tool_call_fn=None):
        self._server = server
        self._tool_call_fn = tool_call_fn
    def list_tools(self):
        if self._server is not None:
            return self._server.list_tools()
        return []
    def call_tool(self, name, args):
        if self._tool_call_fn is not None:
            return self._tool_call_fn(name, args)
        if self._server is not None:
            return self._server.call_tool(name, args)
        return "Error: no server or tool_call_fn configured"
def build_mcp_selection_prompt(query, tools):
    menu = tool_schema_text(tools)
    system = "\n".join([
        "You are a tool router. Pick the best tool for the request.",
        "Available tools:", menu,
        "Return ONLY a JSON object with keys 'tool' and 'args'.",
        "Use tool name 'none' if no tool fits.",
    ])
    return [{"role": "system", "content": system},
            {"role": "user",   "content": "Request: " + str(query)}]

def select_mcp_tool(query, client, llm_fn=None):
    tools = client.list_tools()
    if not tools:
        return {"tool": "none", "args": {}}
    messages = build_mcp_selection_prompt(query, tools)
    response = call_llm(messages, llm_fn=llm_fn)
    data = safe_parse_json(response) or {}
    name = data.get("tool", "none")
    args = data.get("args", {})
    known = {t.name for t in tools}
    if name not in known:
        name = "none"
    return {"tool": name, "args": args if isinstance(args, dict) else {}}
def _mock_mcp_llm(tool="none", args=None):
    payload = json.dumps({"tool": tool, "args": args or {}})
    return lambda messages: payload

def _mock_tool_call(name, args):
    return "Result:" + str(name)

# ── Exercise: implement MCPAgent ─────────────────────────────────────────────

class MCPAgent:
    """An agent that discovers and uses tools from an MCP client."""

    def __init__(self, client, llm_fn=None):
        # TODO: store client, llm_fn, and an empty history list
        pass

    def tools(self):
        # TODO: return self.client.list_tools()
        return []

    def ask(self, query):
        # TODO: call select_mcp_tool(query, self.client, llm_fn=self._llm_fn)
        # If selection["tool"] == "none": result = "No suitable tool found."
        # Else: result = self.client.call_tool(tool, args)
        # Append {"query", "tool", "args", "result"} to self._history
        # Return the record dict
        return {}

    def history(self):
        # TODO: return a copy of self._history
        return []

    def clear_history(self):
        # TODO: clear self._history
        pass


### Checks

In [ ]:
checks = 0

# helper setup
_srv = MCPServer("s")
@_srv.tool("upper", "Uppercase text.", {"text": "str"})
def _upper(text): return str(text).upper()
_client = MCPClient(server=_srv)

# 1 — MCPAgent constructs
try:
    agent = MCPAgent(_client, llm_fn=_mock_mcp_llm("upper", {"text": "hello"}))
    checks += 1; print("✅ 1 MCPAgent constructs")
except Exception as e:
    print("❌ 1:", e)

# 2 — tools() returns client's tool list
try:
    agent = MCPAgent(_client)
    assert any(t.name == "upper" for t in agent.tools())
    checks += 1; print("✅ 2 tools() returns client's tool list")
except Exception as e:
    print("❌ 2:", e)

# 3 — ask() returns dict with tool, args, result
try:
    agent = MCPAgent(_client, llm_fn=_mock_mcp_llm("upper", {"text": "hello"}))
    r = agent.ask("uppercase hello")
    assert r["tool"] == "upper" and r["result"] == "HELLO"
    checks += 1; print("✅ 3 ask() routes and returns result")
except Exception as e:
    print("❌ 3:", e)

# 4 — ask() with no tool returns "No suitable tool found."
try:
    agent = MCPAgent(_client, llm_fn=_mock_mcp_llm("none"))
    r = agent.ask("something unrelated")
    assert r["tool"] == "none" and "No suitable tool" in r["result"]
    checks += 1; print("✅ 4 ask() with no tool returns fallback message")
except Exception as e:
    print("❌ 4:", e)

# 5 — history grows with each ask
try:
    agent = MCPAgent(_client, llm_fn=_mock_mcp_llm("upper", {"text": "x"}))
    agent.ask("q1"); agent.ask("q2")
    assert len(agent.history()) == 2
    checks += 1; print("✅ 5 history() grows with each ask()")
except Exception as e:
    print("❌ 5:", e)

# 6 — clear_history empties history
try:
    agent = MCPAgent(_client, llm_fn=_mock_mcp_llm("upper", {"text": "x"}))
    agent.ask("q")
    agent.clear_history()
    assert agent.history() == []
    checks += 1; print("✅ 6 clear_history() empties history")
except Exception as e:
    print("❌ 6:", e)

print(f"\n{checks}/6 checks passed!")
